# Notebook 16 — Cross-Validation: Zero-Shot Transfer Across All DO Gauges

Tests whether zero-shot transfer works across all training source gauges,
not just gauge 2473. Leave-one-out design: train on each gauge, transfer to all others.

In [1]:
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

REPO_ROOT = Path('/storage/homefs/tn20y076/AareML')
sys.path.insert(0, str(REPO_ROOT))

from src.config import (FEATURES, TARGETS, LOOKBACK, HORIZON,
                         TRAIN_END, VAL_END, SEED, RESULTS_DIR, FIGURES_DIR)
from src.data import load_gauge, preprocess, train_val_test_split, make_windows
from src.model import Seq2SeqLSTM, RiverDataset, train_model, predict, get_y_true
from src.metrics import mean_rmse, nse

np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# Hyperparameters (same as nb03)
HIDDEN, N_LAYERS, DROPOUT = 64, 2, 0.2   # lighter model for LOO speed
LR, EPOCHS, PATIENCE = 1e-3, 30, 5  # reduced for 4h budget

# All 16 DO gauges sorted by DO coverage
DO_GAUGES = [2473, 2009, 2613, 2143, 2016, 2174, 2415,
             2044, 2410, 2085, 2462, 2018, 2243, 2068, 2135, 2130]
print(f'DO gauges: {len(DO_GAUGES)}')

Device: cuda
DO gauges: 16


## 1. Leave-One-Out Cross-Validation

In [2]:
from sklearn.preprocessing import StandardScaler

print('Loading and scaling all gauge data...')
gauge_data = {}

for gid in tqdm(DO_GAUGES, desc='Loading gauges'):
    try:
        raw  = load_gauge(gid)
        data = preprocess(raw)
        train_df, val_df, test_df = train_val_test_split(data)
        train_means = (pd.concat([train_df[FEATURES].mean(), train_df[TARGETS].mean()])
                       .groupby(level=0).first())

        X_tr, y_tr, _ = make_windows(train_df, train_means, features=FEATURES, targets=TARGETS)
        X_va, y_va, _ = make_windows(val_df,   train_means, features=FEATURES, targets=TARGETS)
        X_te, y_te, _ = make_windows(test_df,  train_means, features=FEATURES, targets=TARGETS)

        if len(X_te) < 10:
            print(f'  {gid}: skipped ({len(X_te)} test windows)')
            continue

        N_tr, L, F = X_tr.shape
        H, T = y_tr.shape[1], y_tr.shape[2]

        feat_sc = StandardScaler().fit(X_tr.reshape(-1, F))
        tgt_sc  = StandardScaler().fit(y_tr.reshape(-1, T))

        def scale(X_raw, y_raw, N):
            Xs = feat_sc.transform(X_raw.reshape(-1, F)).reshape(N, L, F).astype('float32')
            ys = tgt_sc.transform(y_raw.reshape(-1, T)).reshape(N, H, T).astype('float32')
            return Xs, ys

        Xs_tr, ys_tr = scale(X_tr, y_tr, N_tr)
        Xs_va, ys_va = scale(X_va, y_va, X_va.shape[0])
        Xs_te, ys_te = scale(X_te, y_te, X_te.shape[0])

        gauge_data[gid] = {
            'ds_tr': RiverDataset(Xs_tr, ys_tr),
            'ds_va': RiverDataset(Xs_va, ys_va),
            'ds_te': RiverDataset(Xs_te, ys_te),
            'tgt_sc': tgt_sc,
        }
        print(f'  {gid}: {N_tr} train / {X_va.shape[0]} val / {X_te.shape[0]} test windows')
    except Exception as e:
        print(f'  {gid}: FAILED ({e})')

print(f'\nLoaded: {len(gauge_data)}/{len(DO_GAUGES)} gauges')


Loading and scaling all gauge data...


Loading gauges:   0%|          | 0/16 [00:00<?, ?it/s]

[data] load_gauge 2473: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.01, 'ec_sensor': 0.007, 'O2C_sensor': 0.012}
[data] split: train=12418, val=731, test=1461


[data] make_windows: 12099 windows, X=(12099, 21, 4), y=(12099, 14, 2), date range 1981-01-22 → 2014-12-18
[data] make_windows: 697 windows, X=(697, 21, 4), y=(697, 14, 2), date range 2015-01-22 → 2016-12-18
[data] make_windows: 1348 windows, X=(1348, 21, 4), y=(1348, 14, 2), date range 2017-01-22 → 2020-12-18
[model] RiverDataset: 12099 samples, X=(12099, 21, 4), y=(12099, 14, 2)
[model] RiverDataset: 697 samples, X=(697, 21, 4), y=(697, 14, 2)
[model] RiverDataset: 1348 samples, X=(1348, 21, 4), y=(1348, 14, 2)
  2473: 12099 train / 697 val / 1348 test windows
[data] load_gauge 2009: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.022, 'ec_sensor': 0.011, 'O2C_sensor': 0.016}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 11975 windows, X=(11975, 21, 4), y=(11975, 14, 2), date range 1981-01-22 → 2014-12-18
[data] make_windows: 697 windows, X=(6

[model] RiverDataset: 11975 samples, X=(11975, 21, 4), y=(11975, 14, 2)
[model] RiverDataset: 697 samples, X=(697, 21, 4), y=(697, 14, 2)
[model] RiverDataset: 1397 samples, X=(1397, 21, 4), y=(1397, 14, 2)
  2009: 11975 train / 697 val / 1397 test windows
[data] load_gauge 2613: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.35, 'pH_sensor': 0.023, 'ec_sensor': 0.008, 'O2C_sensor': 0.016}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 7102 windows, X=(7102, 21, 4), y=(7102, 14, 2), date range 1995-01-01 → 2014-12-18
[data] make_windows: 697 windows, X=(697, 21, 4), y=(697, 14, 2), date range 2015-01-22 → 2016-12-18
[data] make_windows: 1427 windows, X=(1427, 21, 4), y=(1427, 14, 2), date range 2017-01-22 → 2020-12-18
[model] RiverDataset: 7102 samples, X=(7102, 21, 4), y=(7102, 14, 2)
[model] RiverDataset: 697 samples, X=(697, 21, 4), y=(697, 14, 2)
[model] Rive

...

[data] make_windows: 9096 windows, X=(9096, 21, 4), y=(9096, 14, 2), date range 1986-01-29 → 2014-12-18
[data] make_windows: 697 windows, X=(697, 21, 4), y=(697, 14, 2), date range 2015-01-22 → 2016-12-18
[data] make_windows: 1278 windows, X=(1278, 21, 4), y=(1278, 14, 2), date range 2017-01-22 → 2020-12-18
[model] RiverDataset: 9096 samples, X=(9096, 21, 4), y=(9096, 14, 2)
[model] RiverDataset: 697 samples, X=(697, 21, 4), y=(697, 14, 2)
[model] RiverDataset: 1278 samples, X=(1278, 21, 4), y=(1278, 14, 2)
  2044: 9096 train / 697 val / 1278 test windows
[data] load_gauge 2410: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.263, 'pH_sensor': 0.268, 'ec_sensor': 0.263, 'O2C_sensor': 0.265}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 8374 windows, X=(8374, 21, 4), y=(8374, 14, 2), date range 1991-04-12 → 2014-12-18
[data] make_windows: 697 windows, X=(697, 21, 

[model] RiverDataset: 8374 samples, X=(8374, 21, 4), y=(8374, 14, 2)
[model] RiverDataset: 697 samples, X=(697, 21, 4), y=(697, 14, 2)
[model] RiverDataset: 1392 samples, X=(1392, 21, 4), y=(1392, 14, 2)
  2410: 8374 train / 697 val / 1392 test windows
[data] load_gauge 2085: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.455, 'ec_sensor': 0.357, 'O2C_sensor': 0.463}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 5452 windows, X=(5452, 21, 4), y=(5452, 14, 2), date range 1981-01-22 → 2014-12-18
[data] make_windows: 697 windows, X=(697, 21, 4), y=(697, 14, 2), date range 2015-01-22 → 2016-12-18
[data] make_windows: 1394 windows, X=(1394, 21, 4), y=(1394, 14, 2), date range 2017-01-22 → 2020-12-18
[model] RiverDataset: 5452 samples, X=(5452, 21, 4), y=(5452, 14, 2)
[model] RiverDataset: 697 samples, X=(697, 21, 4), y=(697, 14, 2)
[model] RiverData

[data] make_windows: 4416 windows, X=(4416, 21, 4), y=(4416, 14, 2), date range 1999-03-25 → 2014-12-18
[data] make_windows: 697 windows, X=(697, 21, 4), y=(697, 14, 2), date range 2015-01-22 → 2016-12-18
[data] make_windows: 1427 windows, X=(1427, 21, 4), y=(1427, 14, 2), date range 2017-01-22 → 2020-12-18
[model] RiverDataset: 4416 samples, X=(4416, 21, 4), y=(4416, 14, 2)
[model] RiverDataset: 697 samples, X=(697, 21, 4), y=(697, 14, 2)
[model] RiverDataset: 1427 samples, X=(1427, 21, 4), y=(1427, 14, 2)
  2462: 4416 train / 697 val / 1427 test windows
[data] load_gauge 2018: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.576, 'ec_sensor': 0.564, 'O2C_sensor': 0.568}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 5247 windows, X=(5247, 21, 4), y=(5247, 14, 2), date range 1981-01-22 → 2006-01-04
  2018: FAILED (make_windows produced 0 valid wi

[data] load_gauge 2068: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.615, 'ec_sensor': 0.614, 'O2C_sensor': 0.614}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 5103 windows, X=(5103, 21, 4), y=(5103, 14, 2), date range 1981-01-22 → 2014-12-18
[data] make_windows: 351 windows, X=(351, 21, 4), y=(351, 14, 2), date range 2015-01-22 → 2016-01-07
  2068: FAILED (make_windows produced 0 valid windows. Check NaN coverage — the target columns may be entirely missing or the DataFrame is shorter than lookback + horizon = 35 days.)
[data] load_gauge 2135: 14610 rows, 1981-01-01 → 2020-12-31, cols=['temp_sensor', 'pH_sensor', 'ec_sensor', 'O2C_sensor']
[data] preprocess: 14610 days, NaN%={'temp_sensor': 0.0, 'pH_sensor': 0.674, 'ec_sensor': 0.673, 'O2C_sensor': 0.67}
[data] split: train=12418, val=731, test=1461
[data] make_windows: 4728 windows, X=(472

[data] make_windows: 1722 windows, X=(1722, 21, 4), y=(1722, 14, 2), date range 1981-01-22 → 1985-12-25
  2130: FAILED (make_windows produced 0 valid windows. Check NaN coverage — the target columns may be entirely missing or the DataFrame is shorter than lookback + horizon = 35 days.)

Loaded: 11/16 gauges


In [3]:
DO_IDX = list(TARGETS).index('O2C_sensor')
results = []

for src_gid in tqdm(DO_GAUGES, desc='Source gauges'):
    if src_gid not in gauge_data:
        continue

    gd = gauge_data[src_gid]
    dl_tr = torch.utils.data.DataLoader(gd['ds_tr'], batch_size=256, shuffle=True)
    dl_va = torch.utils.data.DataLoader(gd['ds_va'], batch_size=256, shuffle=False)

    model = Seq2SeqLSTM(
        n_feat=len(FEATURES), n_tgt=len(TARGETS),
        hidden=HIDDEN, n_layers=N_LAYERS, dropout=DROPOUT,
    ).to(DEVICE)
    model, _ = train_model(model, dl_tr, dl_va, device=DEVICE,
                            epochs=EPOCHS, lr=LR, patience=PATIENCE, verbose=False)

    for tgt_gid in DO_GAUGES:
        if tgt_gid == src_gid or tgt_gid not in gauge_data:
            continue
        tgt = gauge_data[tgt_gid]
        try:
            # predict() handles DataLoader + inverse_transform internally
            y_pred = predict(model, tgt['ds_te'], tgt['tgt_sc'], device=DEVICE)
            y_true = get_y_true(tgt['ds_te'], tgt['tgt_sc'])

            # DO only (first target index)
            y_pred_do = y_pred[:, :, DO_IDX]
            y_true_do = y_true[:, :, DO_IDX]

            rmse_val = float(np.sqrt(np.mean((y_pred_do - y_true_do)**2)))
            ss_res   = np.sum((y_true_do - y_pred_do)**2)
            ss_tot   = np.sum((y_true_do - y_true_do.mean())**2)
            nse_val  = float(1 - ss_res/ss_tot) if ss_tot > 0 else float('nan')

            results.append({
                'source_gauge': src_gid,
                'target_gauge': tgt_gid,
                'rmse_do': round(rmse_val, 4),
                'nse_do':  round(nse_val,  3),
            })
        except Exception as e:
            print(f'  {src_gid}→{tgt_gid}: {e}')

print(f'\nResults collected: {len(results)} pairs')
df_cv = pd.DataFrame(results) if results else pd.DataFrame(
    columns=['source_gauge','target_gauge','rmse_do','nse_do'])

if not df_cv.empty:
    src_2473 = df_cv[df_cv['source_gauge'] == 2473]
    src_other = df_cv[df_cv['source_gauge'] != 2473]
    print(f'Overall RMSE:   {df_cv["rmse_do"].mean():.4f} mg/L')
    print(f'Overall NSE:    {df_cv["nse_do"].mean():.3f}')
    print(f'Source=2473:    RMSE={src_2473["rmse_do"].mean():.4f}  NSE={src_2473["nse_do"].mean():.3f}')
    print(f'Other sources:  RMSE={src_other["rmse_do"].mean():.4f}  NSE={src_other["nse_do"].mean():.3f}')
    df_cv.to_csv(RESULTS_DIR / 'cv_transfer_results.csv', index=False)
    print(f'Saved: cv_transfer_results.csv')


Source gauges:   0%|          | 0/16 [00:00<?, ?it/s]

[model] train_model: 102,018 trainable params, device=cuda, epochs=30, patience=5, lr=0.001


[model] predict: 1397 samples, DO range [-1.38, 2.31] mg/L (scaled)
[model] predict: 1427 samples, DO range [-1.81, 2.00] mg/L (scaled)
[model] predict: 1412 samples, DO range [-1.96, 2.34] mg/L (scaled)
[model] predict: 1377 samples, DO range [-1.79, 2.23] mg/L (scaled)
[model] predict: 1203 samples, DO range [-1.75, 1.61] mg/L (scaled)
[model] predict: 1374 samples, DO range [-1.77, 2.48] mg/L (scaled)
[model] predict: 1278 samples, DO range [-1.82, 2.28] mg/L (scaled)
[model] predict: 1392 samples, DO range [-2.27, 2.18] mg/L (scaled)
[model] predict: 1394 samples, DO range [-1.82, 2.21] mg/L (scaled)
[model] predict: 1427 samples, DO range [-1.15, 2.24] mg/L (scaled)
[model] train_model: 102,018 trainable params, device=cuda, epochs=30, patience=5, lr=0.001


[model] predict: 1348 samples, DO range [-1.76, 2.34] mg/L (scaled)
[model] predict: 1427 samples, DO range [-1.71, 2.11] mg/L (scaled)
[model] predict: 1412 samples, DO range [-1.83, 2.41] mg/L (scaled)
[model] predict: 1377 samples, DO range [-1.84, 2.25] mg/L (scaled)
[model] predict: 1203 samples, DO range [-1.97, 1.74] mg/L (scaled)
[model] predict: 1374 samples, DO range [-1.83, 2.57] mg/L (scaled)
[model] predict: 1278 samples, DO range [-1.72, 2.28] mg/L (scaled)
[model] predict: 1392 samples, DO range [-2.11, 2.24] mg/L (scaled)
[model] predict: 1394 samples, DO range [-1.85, 2.33] mg/L (scaled)
[model] predict: 1427 samples, DO range [-1.17, 2.28] mg/L (scaled)
[model] train_model: 102,018 trainable params, device=cuda, epochs=30, patience=5, lr=0.001


[model] predict: 1348 samples, DO range [-2.35, 1.91] mg/L (scaled)
[model] predict: 1397 samples, DO range [-1.60, 2.20] mg/L (scaled)
[model] predict: 1412 samples, DO range [-2.24, 2.10] mg/L (scaled)
[model] predict: 1377 samples, DO range [-1.69, 1.96] mg/L (scaled)
[model] predict: 1203 samples, DO range [-2.08, 1.47] mg/L (scaled)
[model] predict: 1374 samples, DO range [-2.06, 2.03] mg/L (scaled)
[model] predict: 1278 samples, DO range [-2.05, 1.80] mg/L (scaled)
[model] predict: 1392 samples, DO range [-2.18, 1.80] mg/L (scaled)
[model] predict: 1394 samples, DO range [-1.85, 1.89] mg/L (scaled)
[model] predict: 1427 samples, DO range [-1.14, 1.92] mg/L (scaled)
[model] train_model: 102,018 trainable params, device=cuda, epochs=30, patience=5, lr=0.001


...

[model] predict: 1348 samples, DO range [-2.04, 2.17] mg/L (scaled)
[model] predict: 1397 samples, DO range [-1.53, 2.24] mg/L (scaled)
[model] predict: 1427 samples, DO range [-1.74, 2.00] mg/L (scaled)
[model] predict: 1412 samples, DO range [-1.84, 2.28] mg/L (scaled)
[model] predict: 1377 samples, DO range [-1.70, 2.23] mg/L (scaled)
[model] predict: 1203 samples, DO range [-1.76, 1.58] mg/L (scaled)
[model] predict: 1374 samples, DO range [-1.76, 2.35] mg/L (scaled)
[model] predict: 1392 samples, DO range [-2.21, 2.14] mg/L (scaled)
[model] predict: 1394 samples, DO range [-1.72, 2.18] mg/L (scaled)
[model] predict: 1427 samples, DO range [-1.17, 2.20] mg/L (scaled)
[model] train_model: 102,018 trainable params, device=cuda, epochs=30, patience=5, lr=0.001


[model] predict: 1348 samples, DO range [-1.59, 1.88] mg/L (scaled)
[model] predict: 1397 samples, DO range [-1.21, 2.07] mg/L (scaled)
[model] predict: 1427 samples, DO range [-1.49, 1.83] mg/L (scaled)
[model] predict: 1412 samples, DO range [-1.49, 2.34] mg/L (scaled)
[model] predict: 1377 samples, DO range [-1.47, 2.24] mg/L (scaled)
[model] predict: 1203 samples, DO range [-1.55, 1.44] mg/L (scaled)
[model] predict: 1374 samples, DO range [-1.63, 2.48] mg/L (scaled)
[model] predict: 1278 samples, DO range [-1.45, 2.33] mg/L (scaled)
[model] predict: 1394 samples, DO range [-1.50, 2.02] mg/L (scaled)
[model] predict: 1427 samples, DO range [-0.93, 2.12] mg/L (scaled)
[model] train_model: 102,018 trainable params, device=cuda, epochs=30, patience=5, lr=0.001


[model] predict: 1348 samples, DO range [-1.66, 2.03] mg/L (scaled)
[model] predict: 1397 samples, DO range [-1.16, 2.07] mg/L (scaled)
[model] predict: 1427 samples, DO range [-1.46, 2.04] mg/L (scaled)
[model] predict: 1412 samples, DO range [-1.69, 2.26] mg/L (scaled)
[model] predict: 1377 samples, DO range [-1.41, 2.16] mg/L (scaled)
[model] predict: 1203 samples, DO range [-1.58, 1.51] mg/L (scaled)
[model] predict: 1374 samples, DO range [-1.57, 2.28] mg/L (scaled)
[model] predict: 1278 samples, DO range [-1.48, 2.16] mg/L (scaled)
[model] predict: 1392 samples, DO range [-1.61, 1.82] mg/L (scaled)
[model] predict: 1427 samples, DO range [-0.98, 2.17] mg/L (scaled)
[model] train_model: 102,018 trainable params, device=cuda, epochs=30, patience=5, lr=0.001


[model] predict: 1348 samples, DO range [-1.31, 1.68] mg/L (scaled)
[model] predict: 1397 samples, DO range [-1.07, 1.68] mg/L (scaled)
[model] predict: 1427 samples, DO range [-1.64, 1.60] mg/L (scaled)
[model] predict: 1412 samples, DO range [-1.49, 1.69] mg/L (scaled)
[model] predict: 1377 samples, DO range [-2.52, 1.59] mg/L (scaled)
[model] predict: 1203 samples, DO range [-2.64, 1.33] mg/L (scaled)
[model] predict: 1374 samples, DO range [-1.30, 1.69] mg/L (scaled)
[model] predict: 1278 samples, DO range [-1.30, 1.45] mg/L (scaled)
[model] predict: 1392 samples, DO range [-2.36, 1.57] mg/L (scaled)
[model] predict: 1394 samples, DO range [-2.22, 1.66] mg/L (scaled)

Results collected: 110 pairs
Overall RMSE:   0.4633 mg/L
Overall NSE:    0.794
Source=2473:    RMSE=0.4807  NSE=0.780
Other sources:  RMSE=0.4615  NSE=0.796


Saved: cv_transfer_results.csv


## 2. Results Heatmap

In [4]:
if df_cv.empty:
    print("No results to plot — check errors above")
else:
    # Pivot to matrix
    pivot = df_cv.pivot(index='source_gauge', columns='target_gauge', values='rmse_do')
    
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn_r',
                vmin=0.2, vmax=1.5, ax=ax,
                cbar_kws={'label': 'RMSE (mg/L DO)'})
    ax.set_title('Zero-Shot Transfer RMSE: Source \u2192 Target Gauge\n(lower = better transfer)',
                 fontsize=13)
    ax.set_xlabel('Target Gauge'); ax.set_ylabel('Source Gauge')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'nb16_cv_heatmap.png', dpi=150)
    plt.close()
    print('Saved: nb16_cv_heatmap.png')

Saved: nb16_cv_heatmap.png


## 3. Per-Source Summary

In [5]:
if df_cv.empty:
    print("No results to plot")
else:
    src_summary = df_cv.groupby('source_gauge')['rmse_do'].mean().sort_values()
    fig, ax = plt.subplots(figsize=(10, 5))
    colors = ['#01696F' if g == 2473 else '#D4D1CA' for g in src_summary.index]
    ax.bar(src_summary.index.astype(str), src_summary.values, color=colors)
    ax.axhline(src_summary.mean(), color='#d62728', linestyle='--', linewidth=1.5,
               label=f'Mean across all sources: {src_summary.mean():.3f} mg/L')
    ax.set_xlabel('Source Gauge'); ax.set_ylabel('Mean Transfer RMSE (mg/L)')
    ax.set_title('Mean Zero-Shot Transfer RMSE by Source Gauge\n(teal = gauge 2473, our chosen training gauge)')
    ax.legend(); plt.xticks(rotation=45); plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'nb16_source_comparison.png', dpi=150)
    plt.close()
    print('Saved: nb16_source_comparison.png')

Saved: nb16_source_comparison.png


## 4. Summary

In [6]:
if df_cv.empty:
    print("No results — check cell 4 errors")
else:
    print('=' * 60)
    print('CROSS-VALIDATION SUMMARY')
    print('=' * 60)
    print(f'Total transfer pairs:     {len(df_cv)}')
    print(f'Overall mean RMSE:        {df_cv["rmse_do"].mean():.4f} mg/L')
    print(f'Overall mean NSE:         {df_cv["nse_do"].mean():.3f}')
    print(f'')
    print(f'Source=2473 mean RMSE:    {src_2473["rmse_do"].mean():.4f} mg/L')
    print(f'All-source mean RMSE:     {df_cv["rmse_do"].mean():.4f} mg/L')
    print(f'')
    print(f'Best source gauge:        {src_summary.index[0]} (RMSE={src_summary.iloc[0]:.4f})')
    print(f'Worst source gauge:       {src_summary.index[-1]} (RMSE={src_summary.iloc[-1]:.4f})')
    print(f'')
    print(f'Interpretation: {"gauge 2473 transfers better than average" if src_2473["rmse_do"].mean() < df_cv["rmse_do"].mean() else "gauge 2473 is representative of average transfer performance"}')

CROSS-VALIDATION SUMMARY
Total transfer pairs:     110
Overall mean RMSE:        0.4633 mg/L
Overall mean NSE:         0.794

Source=2473 mean RMSE:    0.4807 mg/L
All-source mean RMSE:     0.4633 mg/L

Best source gauge:        2415 (RMSE=0.4230)
Worst source gauge:       2462 (RMSE=0.5808)

Interpretation: gauge 2473 is representative of average transfer performance
